## 1. Imports

In [1]:
from scipy.io.arff import loadarff
import pandas as pd
import numpy as np

## 2. Loading the Dataset

The dataset is in `.arff` format, native to the Weka ecosystem.
`loadarff` returns the data and metadata separately; the metadata is discarded
with `_` since it is not used in this pipeline.

In [2]:
data, _ = loadarff('supermarket.arff')

df = pd.DataFrame(np.array(data), dtype=str)

print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Loaded: 4627 rows, 217 columns


,department1,department2,department3,department4,department5,department6,department7,department8,department9,grocery misc,...,department208,department209,department210,department211,department212,department213,department214,department215,department216,total
0,?,?,?,?,?,?,?,?,?,?,...,?,?,?,?,?,?,?,?,?,high
1,t,?,?,?,?,?,?,?,?,?,...,?,?,?,?,?,?,?,?,?,low
2,?,?,?,?,?,?,?,?,?,?,...,?,?,?,?,?,?,?,?,?,low
3,t,?,?,?,?,?,?,?,?,?,...,?,?,?,?,?,?,?,?,?,low
4,?,?,?,?,?,?,?,?,?,?,...,?,?,?,?,?,?,?,?,?,low


## 3. Removing Department Columns

Columns prefixed with `department` are structural metadata from the original
Weka file. They do not represent purchasable items and have no analytical value
for basket mining. They are identified and removed automatically using Pandas
string filtering, avoiding any hardcoded column names.

In [3]:
dept_cols = df.filter(like='department').columns
df = df.drop(columns=dept_cols)

print(f"Removed {len(dept_cols)} department columns.")
print(f"Remaining shape: {df.shape}")

Removed 111 department columns.
Remaining shape: (4627, 106)


## 4. Removing Zero-Support Columns

Columns where every value is `?` indicate items that never appeared in any
transaction — that is, items with support equal to zero. These columns carry
no information and only increase dimensionality unnecessarily.

A boolean mask checks whether **all** values in a column equal `?`. Columns
that satisfy this condition are dropped. An assertion confirms the operation
was successful before proceeding.

In [4]:
empty_cols = df.columns[(df == '?').all()]
df = df.drop(columns=empty_cols)

print(f"Removed {len(empty_cols)} zero-support columns.")
print(f"Final shape: {df.shape}")

assert not (df == '?').all().any(), "Zero-support columns still present after cleaning."
print("Assertion passed: no zero-support columns remaining.")

Removed 5 zero-support columns.
Final shape: (4627, 101)


Assertion passed: no zero-support columns remaining.


## 5. Exporting the Cleaned Dataset

The cleaned DataFrame is exported to CSV for use in Weka. The `index=False`
parameter is required: without it, Pandas writes its integer row index as an
additional column, which Weka would interpret as a valid attribute and include
in rule generation.

In [5]:
output_path = 'supermarket_clean.csv'
df.to_csv(output_path, index=False)

print(f"File exported: {output_path}")
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")

File exported: supermarket_clean.csv
Rows: 4627 | Columns: 101


## 6. Final Preview

A quick inspection of the cleaned dataset before handing it off to Weka.

In [6]:
print("Column list:")
print(df.columns.tolist())

print(f"\nMissing values per column (top 10):")
print((df == '?').sum().sort_values(ascending=False).head(10))

Column list:
['grocery misc', 'baby needs', 'bread and cake', 'baking needs', 'juice-sat-cord-ms', 'tea', 'biscuits', 'canned fish-meat', 'canned fruit', 'canned vegetables', 'breakfast food', 'cigs-tobacco pkts', 'cigarette cartons', 'cleaners-polishers', 'coffee', 'sauces-gravy-pkle', 'confectionary', 'puddings-deserts', 'dishcloths-scour', 'deod-disinfectant', 'frozen foods', 'razor blades', 'fuels-garden aids', 'spices', 'jams-spreads', 'insecticides', 'pet foods', 'laundry needs', 'party snack foods', 'tissues-paper prd', 'wrapping', 'dried vegetables', 'pkt-canned soup', 'soft drinks', 'health food other', 'beverages hot', 'health&beauty misc', 'deodorants-soap', 'mens toiletries', 'medicines', 'haircare', 'dental needs', 'lotions-creams', 'sanitary pads', 'cough-cold-pain', 'meat misc', 'cheese', 'chickens', 'milk-cream', 'cold-meats', 'deli gourmet', 'margarine', 'salads', 'small goods', 'dairy foods', 'fruit drinks', 'delicatessen misc', 'beef', 'hogget', 'lamb', 'pet food', '

gourmet meat         4625
salads               4621
chickens             4606
mutton               4604
sparkling imp        4604
port and sherry      4602
dried vegetables     4598
plants               4598
fruit drinks         4595
cigarette cartons    4590
dtype: int64
